# Day 5: Agentic AI with LangGraph  
## Insurance Claims Workflow — Hands-on Lab Notebook

**Audience:** Beginner to Intermediate UiPath Developers, RPA Developers, Automation Engineers, AI Automation Teams  
**Duration:** 4 Hours  
**Domain:** Insurance Claims Intake, Document Validation, Policy Lookup, Fraud Risk Review and Claim Routing

---

## Learning Outcomes

By the end of this notebook, participants will be able to:

- Understand Agentic AI from an automation engineering perspective.
- Understand LangGraph state, nodes, edges and conditional routing.
- Compare single-agent and multi-agent workflow design.
- Define agent responsibilities and boundaries.
- Decompose an insurance claim workflow into agentic steps.
- Create a claim intake agent.
- Create a document validation agent.
- Create a policy lookup agent.
- Create a fraud risk review agent.
- Route claims based on policy and risk status.
- Build a LangGraph-based Insurance Claim Workflow.

# 0. How to Use This Notebook

This notebook is designed for hands-on practice.

Recommended method:

1. Read each concept explanation.
2. Run the code cell.
3. Observe output.
4. Modify sample claim data.
5. Complete each mini assignment.
6. Build the final LangGraph insurance claim workflow.

Environment variables are assumed to be configured:

```text
OPENROUTER_API_KEY=your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=openai/gpt-4.1-mini
```

Most early labs use deterministic Python logic so participants understand workflow behavior before connecting LLM intelligence.

# 1. Day 5 Session Flow

| Time | Topic |
|---|---|
| 25 min | Agentic AI concepts for automation engineers |
| 35 min | LangGraph state, nodes, edges and conditional routing |
| 35 min | Single-agent vs multi-agent design |
| 35 min | Claim intake and document validation agents |
| 35 min | Policy lookup and fraud risk review agents |
| 35 min | Conditional routing based on policy and risk |
| 40 min | Complete LangGraph insurance claim workflow |
| 25 min | Extra examples, assignments and review |

---

## Trainer Humour

Traditional workflow says:  
“Tell me exactly what to do.”

Agentic workflow says:  
“I know my role, I know the next step, and I promise not to approve claims without evidence.”

# 2. Install Required Libraries

Required packages:

- openai
- python-dotenv
- langgraph
- langchain
- langchain-openai
- typing_extensions

Install only if missing.

In [ ]:
# Uncomment only if packages are missing.
# !pip install openai python-dotenv langgraph langchain langchain-openai typing_extensions

In [ ]:
# pip install langgraph

In [6]:
import langgraph

help(langgraph.graph)

Help on package langgraph.graph in langgraph:

NAME
    langgraph.graph

PACKAGE CONTENTS
    branch
    graph
    message
    schema_utils
    state
    ui

CLASSES
    builtins.dict(builtins.object)
        langgraph.graph.message.MessagesState
    builtins.object
        langgraph.graph.graph.Graph
            langgraph.graph.state.StateGraph
                langgraph.graph.message.MessageGraph
    
    class Graph(builtins.object)
     |  Graph() -> None
     |  
     |  Methods defined here:
     |  
     |  __init__(self) -> None
     |      Initialize self.  See help(type(self)) for accurate signature.
     |  
     |  add_conditional_edges(self, source: str, path: Union[Callable[..., Union[collections.abc.Hashable, list[collections.abc.Hashable]]], Callable[..., collections.abc.Awaitable[Union[collections.abc.Hashable, list[collections.abc.Hashable]]]], langchain_core.runnables.base.Runnable[Any, Union[collections.abc.Hashable, list[collections.abc.Hashable]]]], path_map: Union

# 3. API and LLM Setup

We will use OpenRouter-compatible OpenAI client.

This setup allows the same notebook to call:

- OpenRouter models
- OpenAI-compatible enterprise gateways
- Other compatible providers

The API key is read from environment variables.

In [7]:
import os
import json
from typing import TypedDict, List, Dict, Any, Optional
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

print("Base URL:", OPENROUTER_BASE_URL)
print("Model:", OPENROUTER_MODEL)
print("API Key Loaded:", bool(OPENROUTER_API_KEY))

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY
)

def call_llm(prompt, system_message="You are an insurance automation assistant.", temperature=0.2):
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

Base URL: https://openrouter.ai/api/v1
Model: openai/gpt-4.1-mini
API Key Loaded: True


# 4. What is Agentic AI?

Agentic AI means AI systems that can perform a defined role in a workflow.

An agent can:

- Receive input
- Understand task context
- Use tools or business rules
- Produce output
- Pass state to the next step
- Trigger routing decisions

## Insurance Example

An insurance claim workflow may have these agents:

| Agent | Responsibility |
|---|---|
| Claim Intake Agent | Extract basic claim details |
| Document Validation Agent | Check submitted and missing documents |
| Policy Lookup Agent | Validate policy status and coverage |
| Fraud Risk Review Agent | Identify risk indicators |
| Routing Agent | Decide next queue or action |

Important: Agents should have boundaries.  
An intake agent should not approve claims.  
A fraud review agent should not accuse customers.  
A routing agent should not invent policy data.

# 5. LangGraph Core Concepts

LangGraph helps design stateful agent workflows.

## Important Concepts

| Concept | Meaning |
|---|---|
| State | Shared workflow data |
| Node | A processing step or agent |
| Edge | Connection between nodes |
| Conditional Edge | Route based on decision |
| START | Workflow start |
| END | Workflow end |

## Simple Flow

```text
START
  ↓
Claim Intake Agent
  ↓
Document Validation Agent
  ↓
Policy Lookup Agent
  ↓
Fraud Risk Review Agent
  ↓
Routing Decision
  ↓
END
```

# 6. Sample Insurance Claim Data

We will use these sample claims throughout the notebook.

In [2]:
health_claim = {
    "claim_id": "CLM9001",
    "policy_number": "POL1001",
    "customer_name": "Rajesh Sharma",
    "claim_type": "Health",
    "claim_amount": 85000,
    "claim_description": "Hospitalization due to dengue fever at City Care Hospital.",
    "submitted_documents": ["Hospital Bill", "Discharge Summary", "Doctor Prescription"],
    "incident_date": "2026-06-12"
}

vehicle_claim = {
    "claim_id": "CLM9002",
    "policy_number": "VEH2045",
    "customer_name": "Meena Joshi",
    "claim_type": "Vehicle",
    "claim_amount": 240000,
    "claim_description": "Vehicle accident at midnight. Repair estimate appears high. Photos are unclear.",
    "submitted_documents": ["Repair Estimate", "Vehicle Photos"],
    "incident_date": "2026-06-18"
}

travel_claim = {
    "claim_id": "CLM9003",
    "policy_number": "TRV3001",
    "customer_name": "Amit Verma",
    "claim_type": "Travel",
    "claim_amount": 18000,
    "claim_description": "Baggage delayed by 48 hours during international travel.",
    "submitted_documents": ["Boarding Pass", "Airline Delay Certificate"],
    "incident_date": "2026-06-20"
}

print("Sample claims loaded.")

Sample claims loaded.


# 7. Insurance Reference Data

In enterprise automation, agents may query:

- Policy system
- Customer system
- Claim system
- Payment system
- Fraud history system
- Document management system

For practice, we will use local dictionaries as mock enterprise systems.

In [3]:
policy_database = {
    "POL1001": {
        "policy_type": "Health",
        "policy_status": "Active",
        "premium_status": "Paid",
        "coverage_amount": 500000,
        "customer_name": "Rajesh Sharma"
    },
    "VEH2045": {
        "policy_type": "Vehicle",
        "policy_status": "Active",
        "premium_status": "Pending",
        "coverage_amount": 300000,
        "customer_name": "Meena Joshi"
    },
    "TRV3001": {
        "policy_type": "Travel",
        "policy_status": "Active",
        "premium_status": "Paid",
        "coverage_amount": 100000,
        "customer_name": "Amit Verma"
    }
}

required_documents_by_claim_type = {
    "Health": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],
    "Vehicle": ["Repair Estimate", "Vehicle Photos", "Police Report", "Driving License", "RC Copy"],
    "Travel": ["Boarding Pass", "Airline Delay Certificate", "Baggage Delay Report", "Claim Form"],
    "Property": ["Damage Photos", "Repair Estimate", "Ownership Proof", "Claim Form"]
}

fraud_keywords = [
    "midnight",
    "unclear",
    "high estimate",
    "policy started recently",
    "missing police report",
    "unusually high"
]

print("Reference data loaded.")

Reference data loaded.


# 8. Define LangGraph State

State is the shared memory of the workflow.

Every agent reads from state and writes back to state.

## Insurance Claim State

Our state will store:

- claim data
- intake summary
- missing documents
- policy details
- policy validation status
- fraud risk level
- routing decision
- workflow logs

In [4]:
class ClaimWorkflowState(TypedDict, total=False):
    claim: Dict[str, Any]
    intake_summary: str
    required_documents: List[str]
    missing_documents: List[str]
    document_status: str
    policy_details: Dict[str, Any]
    policy_status: str
    coverage_status: str
    risk_indicators: List[str]
    risk_level: str
    routing_decision: str
    final_message: str
    logs: List[str]

# 9. Lab 1: Create Claim Intake Agent

## Responsibility

The Claim Intake Agent should:

- Read incoming claim data.
- Extract key fields.
- Create an intake summary.
- Add a workflow log.

## Boundary

It should not approve, reject or classify fraud.

In [5]:
def claim_intake_agent(state: ClaimWorkflowState) -> ClaimWorkflowState:
    claim = state["claim"]

    summary = (
        f"Claim {claim.get('claim_id')} received from {claim.get('customer_name')} "
        f"for {claim.get('claim_type')} insurance. Claim amount: INR {claim.get('claim_amount')}."
    )

    logs = state.get("logs", [])
    logs.append("Claim Intake Agent completed.")

    return {
        **state,
        "intake_summary": summary,
        "logs": logs
    }


test_state = {"claim": health_claim, "logs": []}
result_state = claim_intake_agent(test_state)
print(result_state["intake_summary"])
print(result_state["logs"])

Claim CLM9001 received from Rajesh Sharma for Health insurance. Claim amount: INR 85000.
['Claim Intake Agent completed.']


## Mini Assignment 1

Enhance the Claim Intake Agent to also extract:

- incident_date
- claim_description
- number of submitted documents

Add these details to the intake summary.

# 10. Lab 2: Create Document Validation Agent

## Responsibility

The Document Validation Agent should:

- Identify required documents for claim type.
- Compare with submitted documents.
- Find missing documents.
- Set document status.

## Boundary

It should not reject the claim.  
It only reports document completeness.

In [6]:
def document_validation_agent(state: ClaimWorkflowState) -> ClaimWorkflowState:
    claim = state["claim"]
    claim_type = claim.get("claim_type")
    submitted_documents = claim.get("submitted_documents", [])

    required_documents = required_documents_by_claim_type.get(claim_type, [])
    missing_documents = [
        doc for doc in required_documents
        if doc not in submitted_documents
    ]

    document_status = "Complete" if not missing_documents else "Missing Documents"

    logs = state.get("logs", [])
    logs.append("Document Validation Agent completed.")

    return {
        **state,
        "required_documents": required_documents,
        "missing_documents": missing_documents,
        "document_status": document_status,
        "logs": logs
    }


test_state = {
    "claim": vehicle_claim,
    "logs": []
}
test_state = document_validation_agent(test_state)

print("Required:", test_state["required_documents"])
print("Missing:", test_state["missing_documents"])
print("Status:", test_state["document_status"])

Required: ['Repair Estimate', 'Vehicle Photos', 'Police Report', 'Driving License', 'RC Copy']
Missing: ['Police Report', 'Driving License', 'RC Copy']
Status: Missing Documents


## Mini Assignment 2

Modify the Document Validation Agent to add severity:

- Low: 0 missing documents
- Medium: 1–2 missing documents
- High: more than 2 missing documents

Store result as `document_severity`.

# 11. Lab 3: Create Policy Lookup Agent

## Responsibility

The Policy Lookup Agent should:

- Find policy details.
- Check if policy exists.
- Check active/inactive status.
- Check premium status.
- Check whether claim amount is within coverage.

## Boundary

It should not perform fraud review.

In [7]:
def policy_lookup_agent(state: ClaimWorkflowState) -> ClaimWorkflowState:
    claim = state["claim"]
    policy_number = claim.get("policy_number")
    claim_amount = claim.get("claim_amount", 0)

    policy_details = policy_database.get(policy_number)

    if not policy_details:
        policy_status = "Policy Not Found"
        coverage_status = "Cannot Validate"
    elif policy_details["policy_status"] != "Active":
        policy_status = "Policy Inactive"
        coverage_status = "Cannot Validate"
    elif policy_details["premium_status"] != "Paid":
        policy_status = "Premium Pending"
        coverage_status = "Cannot Validate"
    elif claim_amount > policy_details["coverage_amount"]:
        policy_status = "Policy Valid"
        coverage_status = "Claim Exceeds Coverage"
    else:
        policy_status = "Policy Valid"
        coverage_status = "Within Coverage"

    logs = state.get("logs", [])
    logs.append("Policy Lookup Agent completed.")

    return {
        **state,
        "policy_details": policy_details or {},
        "policy_status": policy_status,
        "coverage_status": coverage_status,
        "logs": logs
    }


test_state = {"claim": health_claim, "logs": []}
test_state = policy_lookup_agent(test_state)

print("Policy Status:", test_state["policy_status"])
print("Coverage Status:", test_state["coverage_status"])
print("Policy Details:", test_state["policy_details"])

Policy Status: Policy Valid
Coverage Status: Within Coverage
Policy Details: {'policy_type': 'Health', 'policy_status': 'Active', 'premium_status': 'Paid', 'coverage_amount': 500000, 'customer_name': 'Rajesh Sharma'}


## Mini Assignment 3

Enhance the Policy Lookup Agent to check customer name mismatch.

If claim customer name and policy customer name are different, store:

```python
customer_match_status = "Mismatch"
```

Otherwise:

```python
customer_match_status = "Matched"
```

# 12. Lab 4: Create Fraud Risk Review Agent

## Responsibility

The Fraud Risk Review Agent should:

- Identify risk indicators.
- Assign risk level.
- Use neutral language.
- Recommend review, not accuse customer.

## Boundary

It should not use words like “fraud confirmed”.  
It should say “risk indicators found”.

In [8]:
def fraud_risk_review_agent(state: ClaimWorkflowState) -> ClaimWorkflowState:
    claim = state["claim"]
    description = claim.get("claim_description", "").lower()
    claim_amount = claim.get("claim_amount", 0)

    risk_indicators = []

    for keyword in fraud_keywords:
        if keyword in description:
            risk_indicators.append(keyword)

    if claim_amount > 200000:
        risk_indicators.append("High claim amount")

    if state.get("document_status") == "Missing Documents":
        risk_indicators.append("Required documents missing")

    if state.get("policy_status") == "Premium Pending":
        risk_indicators.append("Premium pending")

    if len(risk_indicators) >= 3:
        risk_level = "High"
    elif len(risk_indicators) >= 1:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    logs = state.get("logs", [])
    logs.append("Fraud Risk Review Agent completed.")

    return {
        **state,
        "risk_indicators": risk_indicators,
        "risk_level": risk_level,
        "logs": logs
    }


test_state = {"claim": vehicle_claim, "logs": []}
test_state = document_validation_agent(test_state)
test_state = policy_lookup_agent(test_state)
test_state = fraud_risk_review_agent(test_state)

print("Risk Indicators:", test_state["risk_indicators"])
print("Risk Level:", test_state["risk_level"])

Risk Indicators: ['midnight', 'unclear', 'High claim amount', 'Required documents missing', 'Premium pending']
Risk Level: High


## Mini Assignment 4

Enhance the risk logic:

- If policy is very new, add risk indicator.
- If claim amount is more than 80% of coverage, add risk indicator.
- If police report is missing for vehicle claim, add risk indicator.

# 13. Lab 5: Routing Logic

## Responsibility

The Routing Agent decides next step based on:

- document status
- policy status
- coverage status
- risk level

## Possible Routes

| Condition | Route |
|---|---|
| Missing documents | Document Pending Queue |
| Policy invalid | Policy Exception Queue |
| High risk | Risk Review Queue |
| Within coverage + low risk | Auto Processing Queue |
| Other cases | Manual Review Queue |

In [9]:
def route_claim(state: ClaimWorkflowState) -> str:
    if state.get("document_status") == "Missing Documents":
        return "document_pending"

    if state.get("policy_status") != "Policy Valid":
        return "policy_exception"

    if state.get("coverage_status") == "Claim Exceeds Coverage":
        return "manual_review"

    if state.get("risk_level") == "High":
        return "risk_review"

    if state.get("risk_level") == "Low" and state.get("coverage_status") == "Within Coverage":
        return "auto_processing"

    return "manual_review"


def routing_agent(state: ClaimWorkflowState) -> ClaimWorkflowState:
    route = route_claim(state)

    route_map = {
        "document_pending": "Document Pending Queue",
        "policy_exception": "Policy Exception Queue",
        "risk_review": "Risk Review Queue",
        "auto_processing": "Auto Processing Queue",
        "manual_review": "Manual Review Queue"
    }

    logs = state.get("logs", [])
    logs.append("Routing Agent completed.")

    return {
        **state,
        "routing_decision": route_map.get(route, "Manual Review Queue"),
        "logs": logs
    }


test_state = {"claim": vehicle_claim, "logs": []}
for agent in [claim_intake_agent, document_validation_agent, policy_lookup_agent, fraud_risk_review_agent, routing_agent]:
    test_state = agent(test_state)

print("Routing Decision:", test_state["routing_decision"])
print("Logs:", test_state["logs"])

Routing Decision: Document Pending Queue
Logs: ['Claim Intake Agent completed.', 'Document Validation Agent completed.', 'Policy Lookup Agent completed.', 'Fraud Risk Review Agent completed.', 'Routing Agent completed.']


## Mini Assignment 5

Create one new route:

```text
Senior Claim Review Queue
```

Condition:

- Claim amount greater than INR 5,00,000
- Risk level is Medium or High

# 14. Build First LangGraph Workflow

Now we will connect nodes using LangGraph.

Flow:

```text
START
  ↓
claim_intake
  ↓
document_validation
  ↓
policy_lookup
  ↓
fraud_review
  ↓
routing
  ↓
END
```

In [10]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ClaimWorkflowState)

workflow.add_node("claim_intake", claim_intake_agent)
workflow.add_node("document_validation", document_validation_agent)
workflow.add_node("policy_lookup", policy_lookup_agent)
workflow.add_node("fraud_review", fraud_risk_review_agent)
workflow.add_node("routing", routing_agent)

workflow.add_edge(START, "claim_intake")
workflow.add_edge("claim_intake", "document_validation")
workflow.add_edge("document_validation", "policy_lookup")
workflow.add_edge("policy_lookup", "fraud_review")
workflow.add_edge("fraud_review", "routing")
workflow.add_edge("routing", END)

insurance_claim_app = workflow.compile()

print("LangGraph insurance claim workflow compiled successfully.")

LangGraph insurance claim workflow compiled successfully.


# 15. Run LangGraph Workflow

Let us test the workflow with health and vehicle claims.

In [11]:
initial_state = {
    "claim": health_claim,
    "logs": []
}

final_state = insurance_claim_app.invoke(initial_state)

print("Final Routing Decision:", final_state["routing_decision"])
print("Policy Status:", final_state["policy_status"])
print("Risk Level:", final_state["risk_level"])
print("Missing Documents:", final_state["missing_documents"])
print("Logs:", final_state["logs"])

Final Routing Decision: Document Pending Queue
Policy Status: Policy Valid
Risk Level: Medium
Missing Documents: ['Claim Form']
Logs: ['Claim Intake Agent completed.', 'Document Validation Agent completed.', 'Policy Lookup Agent completed.', 'Fraud Risk Review Agent completed.', 'Routing Agent completed.']


In [12]:
initial_state = {
    "claim": vehicle_claim,
    "logs": []
}

final_state = insurance_claim_app.invoke(initial_state)

print("Final Routing Decision:", final_state["routing_decision"])
print("Policy Status:", final_state["policy_status"])
print("Risk Level:", final_state["risk_level"])
print("Missing Documents:", final_state["missing_documents"])
print("Risk Indicators:", final_state["risk_indicators"])

Final Routing Decision: Document Pending Queue
Policy Status: Premium Pending
Risk Level: High
Missing Documents: ['Police Report', 'Driving License', 'RC Copy']
Risk Indicators: ['midnight', 'unclear', 'High claim amount', 'Required documents missing', 'Premium pending']


## Mini Assignment 6

Run the workflow for `travel_claim`.

Observe:

- document status
- policy status
- risk level
- routing decision

Then modify `travel_claim` by removing one document and rerun.

In [13]:
initial_state = {
    "claim": travel_claim,
    "logs": []
}

final_state = insurance_claim_app.invoke(initial_state)

print("Final Routing Decision:", final_state["routing_decision"])
print("Policy Status:", final_state["policy_status"])
print("Risk Level:", final_state["risk_level"])
print("Missing Documents:", final_state["missing_documents"])

Final Routing Decision: Document Pending Queue
Policy Status: Policy Valid
Risk Level: Medium
Missing Documents: ['Baggage Delay Report', 'Claim Form']


# 16. Conditional Routing in LangGraph

In earlier workflow, routing decision was stored in state but all paths ended.

Now we create conditional edges.

Possible next nodes:

- request_documents
- policy_exception
- risk_review_queue
- auto_processing
- manual_review

In [14]:
def request_documents_node(state: ClaimWorkflowState) -> ClaimWorkflowState:
    missing = state.get("missing_documents", [])
    message = f"Request customer to submit missing documents: {', '.join(missing)}"
    logs = state.get("logs", [])
    logs.append("Request Documents Node completed.")
    return {**state, "final_message": message, "logs": logs}

def policy_exception_node(state: ClaimWorkflowState) -> ClaimWorkflowState:
    message = f"Route to policy exception team. Reason: {state.get('policy_status')}"
    logs = state.get("logs", [])
    logs.append("Policy Exception Node completed.")
    return {**state, "final_message": message, "logs": logs}

def risk_review_node(state: ClaimWorkflowState) -> ClaimWorkflowState:
    message = f"Route to risk review. Indicators: {state.get('risk_indicators')}"
    logs = state.get("logs", [])
    logs.append("Risk Review Node completed.")
    return {**state, "final_message": message, "logs": logs}

def auto_processing_node(state: ClaimWorkflowState) -> ClaimWorkflowState:
    message = "Claim is ready for auto-processing based on current rules."
    logs = state.get("logs", [])
    logs.append("Auto Processing Node completed.")
    return {**state, "final_message": message, "logs": logs}

def manual_review_node(state: ClaimWorkflowState) -> ClaimWorkflowState:
    message = "Route claim to manual review team."
    logs = state.get("logs", [])
    logs.append("Manual Review Node completed.")
    return {**state, "final_message": message, "logs": logs}

In [15]:
conditional_workflow = StateGraph(ClaimWorkflowState)

conditional_workflow.add_node("claim_intake", claim_intake_agent)
conditional_workflow.add_node("document_validation", document_validation_agent)
conditional_workflow.add_node("policy_lookup", policy_lookup_agent)
conditional_workflow.add_node("fraud_review", fraud_risk_review_agent)

conditional_workflow.add_node("request_documents", request_documents_node)
conditional_workflow.add_node("policy_exception", policy_exception_node)
conditional_workflow.add_node("risk_review_queue", risk_review_node)
conditional_workflow.add_node("auto_processing", auto_processing_node)
conditional_workflow.add_node("manual_review", manual_review_node)

conditional_workflow.add_edge(START, "claim_intake")
conditional_workflow.add_edge("claim_intake", "document_validation")
conditional_workflow.add_edge("document_validation", "policy_lookup")
conditional_workflow.add_edge("policy_lookup", "fraud_review")

conditional_workflow.add_conditional_edges(
    "fraud_review",
    route_claim,
    {
        "document_pending": "request_documents",
        "policy_exception": "policy_exception",
        "risk_review": "risk_review_queue",
        "auto_processing": "auto_processing",
        "manual_review": "manual_review"
    }
)

conditional_workflow.add_edge("request_documents", END)
conditional_workflow.add_edge("policy_exception", END)
conditional_workflow.add_edge("risk_review_queue", END)
conditional_workflow.add_edge("auto_processing", END)
conditional_workflow.add_edge("manual_review", END)

conditional_claim_app = conditional_workflow.compile()

print("Conditional LangGraph workflow compiled successfully.")

Conditional LangGraph workflow compiled successfully.


In [16]:
final_state = conditional_claim_app.invoke({
    "claim": vehicle_claim,
    "logs": []
})

print("Final Message:", final_state["final_message"])
print("Risk Level:", final_state["risk_level"])
print("Missing Documents:", final_state["missing_documents"])
print("Logs:", final_state["logs"])

Final Message: Request customer to submit missing documents: Police Report, Driving License, RC Copy
Risk Level: High
Missing Documents: ['Police Report', 'Driving License', 'RC Copy']
Logs: ['Claim Intake Agent completed.', 'Document Validation Agent completed.', 'Policy Lookup Agent completed.', 'Fraud Risk Review Agent completed.', 'Request Documents Node completed.']


# 17. Single-Agent vs Multi-Agent Design

## Single-Agent Design

One agent performs many tasks.

Advantages:

- Simple
- Fewer components
- Easy for small demos

Limitations:

- Hard to debug
- Hard to govern
- Too much responsibility in one place

## Multi-Agent Design

Each agent has a clear responsibility.

Advantages:

- Modular
- Easier to test
- Better governance
- Better auditability
- Similar to enterprise workflow design

For insurance, multi-agent design is usually better because claim processes are governed, auditable and role-based.

# 18. Example 1: LLM-Powered Claim Intake Agent

Now we add LLM intelligence to generate a better intake summary.

This is optional but useful for unstructured claim notes.

In [18]:
def llm_claim_intake_agent(state: ClaimWorkflowState) -> ClaimWorkflowState:
    claim = state["claim"]

    prompt = f"""
You are a claim intake assistant.

Create a concise claim intake summary.

Include:
- claim id
- customer name
- claim type
- claim amount
- incident summary
- submitted document count

Claim:
{json.dumps(claim, indent=2)}
"""
    summary = call_llm(prompt)

    logs = state.get("logs", [])
    logs.append("LLM Claim Intake Agent completed.")

    return {
        **state,
        "intake_summary": summary,
        "logs": logs
    }

# Uncomment after API key setup.
test_state = {"claim": health_claim, "logs": []}
result = llm_claim_intake_agent(test_state)
print(result["intake_summary"])

Claim Intake Summary:
- Claim ID: CLM9001
- Customer Name: Rajesh Sharma
- Claim Type: Health
- Claim Amount: 85,000
- Incident Summary: Hospitalization due to dengue fever at City Care Hospital on 2026-06-12.
- Submitted Document Count: 3


## Assignment Example 1

Modify the LLM Claim Intake Agent to return structured JSON with:

- claim_id
- customer_name
- claim_type
- amount
- short_summary
- immediate_next_step

# 19. Example 2: Health Claim Workflow

Use case:

A health claim should validate hospital documents and policy coverage.

In [19]:
health_claim_incomplete = {
    "claim_id": "CLM9004",
    "policy_number": "POL1001",
    "customer_name": "Rajesh Sharma",
    "claim_type": "Health",
    "claim_amount": 125000,
    "claim_description": "Hospitalization for surgery at City Care Hospital.",
    "submitted_documents": ["Hospital Bill"],
    "incident_date": "2026-06-22"
}

final_state = conditional_claim_app.invoke({
    "claim": health_claim_incomplete,
    "logs": []
})

print("Final Message:", final_state["final_message"])
print("Missing Documents:", final_state["missing_documents"])
print("Routing Decision:", route_claim(final_state))

Final Message: Request customer to submit missing documents: Discharge Summary, Doctor Prescription, Claim Form
Missing Documents: ['Discharge Summary', 'Doctor Prescription', 'Claim Form']
Routing Decision: document_pending


## Assignment Example 2

Add health-specific rule:

If claim type is Health and discharge summary is missing, route to:

```text
Health Document Review Queue
```

# 20. Example 3: Vehicle Claim Workflow

Use case:

Vehicle claims require police report, photos, repair estimate and license documents.

In [20]:
vehicle_claim_high_risk = {
    "claim_id": "CLM9005",
    "policy_number": "VEH2045",
    "customer_name": "Meena Joshi",
    "claim_type": "Vehicle",
    "claim_amount": 280000,
    "claim_description": "Accident happened at midnight. Photos are unclear. Repair estimate unusually high.",
    "submitted_documents": ["Repair Estimate", "Vehicle Photos"],
    "incident_date": "2026-06-23"
}

final_state = conditional_claim_app.invoke({
    "claim": vehicle_claim_high_risk,
    "logs": []
})

print("Final Message:", final_state["final_message"])
print("Risk Indicators:", final_state["risk_indicators"])
print("Risk Level:", final_state["risk_level"])
print("Policy Status:", final_state["policy_status"])

Final Message: Request customer to submit missing documents: Police Report, Driving License, RC Copy
Risk Indicators: ['midnight', 'unclear', 'unusually high', 'High claim amount', 'Required documents missing', 'Premium pending']
Risk Level: High
Policy Status: Premium Pending


## Assignment Example 3

Enhance vehicle claim risk rules:

- Missing police report = Medium risk
- Missing driving license = Medium risk
- Missing both = High risk

# 21. Example 4: Travel Claim Workflow

Use case:

Travel claim may involve baggage delay, flight delay or medical emergency during travel.

In [21]:
travel_claim_missing = {
    "claim_id": "CLM9006",
    "policy_number": "TRV3001",
    "customer_name": "Amit Verma",
    "claim_type": "Travel",
    "claim_amount": 18000,
    "claim_description": "Baggage delayed for 48 hours during international travel.",
    "submitted_documents": ["Boarding Pass"],
    "incident_date": "2026-06-20"
}

final_state = conditional_claim_app.invoke({
    "claim": travel_claim_missing,
    "logs": []
})

print("Final Message:", final_state["final_message"])
print("Missing Documents:", final_state["missing_documents"])
print("Risk Level:", final_state["risk_level"])

Final Message: Request customer to submit missing documents: Airline Delay Certificate, Baggage Delay Report, Claim Form
Missing Documents: ['Airline Delay Certificate', 'Baggage Delay Report', 'Claim Form']
Risk Level: Medium


## Assignment Example 4

Add travel-specific rule:

If baggage delay report is missing, route to:

```text
Travel Document Pending Queue
```

# 22. Example 5: Property Claim Workflow

Use case:

Property claims require damage photos, repair estimate, ownership proof and claim form.

In [22]:
property_claim = {
    "claim_id": "CLM9007",
    "policy_number": "PROP5001",
    "customer_name": "Neha Kulkarni",
    "claim_type": "Property",
    "claim_amount": 350000,
    "claim_description": "Water leakage damaged home interior and furniture.",
    "submitted_documents": ["Damage Photos", "Repair Estimate"],
    "incident_date": "2026-06-25"
}

final_state = conditional_claim_app.invoke({
    "claim": property_claim,
    "logs": []
})

print("Final Message:", final_state["final_message"])
print("Policy Status:", final_state["policy_status"])
print("Missing Documents:", final_state["missing_documents"])

Final Message: Request customer to submit missing documents: Ownership Proof, Claim Form
Policy Status: Policy Not Found
Missing Documents: ['Ownership Proof', 'Claim Form']


## Assignment Example 5

Add property policy data to `policy_database`.

Then rerun the property claim workflow and observe new routing behavior.

# 23. Batch Processing Multiple Claims

Enterprise automation often processes many claims from a queue.

We will process multiple claim dictionaries through LangGraph.

In [23]:
claims_batch = [
    health_claim,
    vehicle_claim,
    travel_claim,
    health_claim_incomplete,
    vehicle_claim_high_risk,
    travel_claim_missing,
    property_claim
]

batch_results = []

for claim in claims_batch:
    final_state = conditional_claim_app.invoke({
        "claim": claim,
        "logs": []
    })

    batch_results.append({
        "claim_id": claim["claim_id"],
        "claim_type": claim["claim_type"],
        "policy_status": final_state.get("policy_status"),
        "document_status": final_state.get("document_status"),
        "risk_level": final_state.get("risk_level"),
        "route": route_claim(final_state),
        "final_message": final_state.get("final_message")
    })

for result in batch_results:
    print(result)

{'claim_id': 'CLM9001', 'claim_type': 'Health', 'policy_status': 'Policy Valid', 'document_status': 'Missing Documents', 'risk_level': 'Medium', 'route': 'document_pending', 'final_message': 'Request customer to submit missing documents: Claim Form'}
{'claim_id': 'CLM9002', 'claim_type': 'Vehicle', 'policy_status': 'Premium Pending', 'document_status': 'Missing Documents', 'risk_level': 'High', 'route': 'document_pending', 'final_message': 'Request customer to submit missing documents: Police Report, Driving License, RC Copy'}
{'claim_id': 'CLM9003', 'claim_type': 'Travel', 'policy_status': 'Policy Valid', 'document_status': 'Missing Documents', 'risk_level': 'Medium', 'route': 'document_pending', 'final_message': 'Request customer to submit missing documents: Baggage Delay Report, Claim Form'}
{'claim_id': 'CLM9004', 'claim_type': 'Health', 'policy_status': 'Policy Valid', 'document_status': 'Missing Documents', 'risk_level': 'Medium', 'route': 'document_pending', 'final_message': 'Re

## Mini Assignment 7

Convert `batch_results` into a pandas DataFrame and export as:

```text
claim_routing_report.csv
```

This report can be consumed by UiPath or business users.

In [ ]:
# Optional assignment solution

import pandas as pd

# df = pd.DataFrame(batch_results)

# df.to_csv("claim_routing_report.csv", index=False)

# df

,claim_id,claim_type,policy_status,document_status,risk_level,route,final_message
0,CLM9001,Health,Policy Valid,Missing Documents,Medium,document_pending,Request customer to submit missing documents: ...
1,CLM9002,Vehicle,Premium Pending,Missing Documents,High,document_pending,Request customer to submit missing documents: ...
2,CLM9003,Travel,Policy Valid,Missing Documents,Medium,document_pending,Request customer to submit missing documents: ...
3,CLM9004,Health,Policy Valid,Missing Documents,Medium,document_pending,Request customer to submit missing documents: ...
4,CLM9005,Vehicle,Premium Pending,Missing Documents,High,document_pending,Request customer to submit missing documents: ...
5,CLM9006,Travel,Policy Valid,Missing Documents,Medium,document_pending,Request customer to submit missing documents: ...
6,CLM9007,Property,Policy Not Found,Missing Documents,Medium,document_pending,Request customer to submit missing documents: ...


# 24. Final Assignment: LangGraph-Based Insurance Claim Workflow

## Assignment Objective

Build a LangGraph-based Insurance Claim Workflow.

## Required Agents

| Agent | Required Output |
|---|---|
| Claim Intake Agent | Intake summary |
| Document Validation Agent | Required and missing documents |
| Policy Lookup Agent | Policy status and coverage status |
| Fraud Risk Review Agent | Risk indicators and risk level |
| Routing Agent | Final queue or next action |

## Required Routes

| Route | Condition |
|---|---|
| Document Pending Queue | Missing documents |
| Policy Exception Queue | Policy not found, inactive or premium pending |
| Risk Review Queue | High risk |
| Auto Processing Queue | Valid policy, complete documents, low risk |
| Manual Review Queue | Default route |

## Deliverables

Participants must submit:

1. LangGraph workflow notebook
2. At least 3 test claim examples
3. Final routing output
4. Workflow logs
5. Short explanation of each agent responsibility

# 25. Final Assignment Checklist

| Requirement | Completed |
|---|---|
| State schema created | ☐ |
| Claim Intake Agent created | ☐ |
| Document Validation Agent created | ☐ |
| Policy Lookup Agent created | ☐ |
| Fraud Risk Review Agent created | ☐ |
| Routing logic created | ☐ |
| LangGraph workflow compiled | ☐ |
| Conditional routing implemented | ☐ |
| Health claim tested | ☐ |
| Vehicle claim tested | ☐ |
| Travel claim tested | ☐ |
| Batch processing tested | ☐ |
| Final output reviewed | ☐ |

# 26. Review and Discussion Questions

1. Why is state important in LangGraph?
2. What is the difference between a node and an agent?
3. Why should each agent have clear responsibility boundaries?
4. Why should fraud review use neutral language?
5. Which claims should go for human review?
6. How can UiPath call this workflow through FastAPI?
7. What should be logged for audit?
8. How can this workflow be extended using MCP tools?
9. How can this workflow connect to actual insurance systems?

# 27. Day 5 Summary

You have completed:

- Agentic AI concepts for automation engineers
- LangGraph state, nodes, edges and conditional routing
- Single-agent versus multi-agent design
- Agent responsibility boundaries
- Insurance claim workflow decomposition
- Claim intake agent
- Document validation agent
- Policy lookup agent
- Fraud risk review agent
- Routing logic
- Conditional LangGraph workflow
- Insurance-specific practice examples
- Batch claim processing
- Final LangGraph assignment

## Next Step

This workflow can be extended with:

- FastAPI deployment
- UiPath HTTP integration
- RAG-based policy lookup
- MCP tools
- Real claim system APIs
- Human-in-the-loop review